# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method choice:** My lane's question ("which pages should an editor review first?") is a yes/no label problem evaluated as a ranking (precision@K) — the same shape as the Week-4 baseline, per the training-honest-models skill's toolkit table. I'm training two models, in order of complexity:

1. **Logistic Regression** — readable coefficients, a fast honest first check that the safe features carry real signal.
2. **Random Forest** — can capture interactions the linear model can't (e.g. staleness only mattering *when* a page still has traffic), and gives permutation-style feature importances for Section 4.

I'm skipping Gradient Boosting for now — the skill says simplicity is a feature, and there's no evidence yet that RF's ceiling is too low to justify a heavier model.

**Features used (identical safe set from my Week-3 data contract):** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `engagement_rate`, `ai_traffic_pct`, `content_age_days`, `days_since_last_update`, `word_count`, `content_type`, `main_intent`. `trend_direction` and `trend_pct` are used ONLY to build the label, never as inputs — same rule as the baseline.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42  # fixed and stated, per the reproducibility basics in training-honest-models

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Missingness follows content_type (checked in ML-04/06) -- impute word_count WITHIN
# content_type, not with a single global median, so we don't inject a content_type signal
df["word_count"] = df.groupby("content_type")["word_count"].transform(lambda s: s.fillna(s.median()))
df["main_intent"] = df["main_intent"].fillna("unknown")

numeric_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                     "engagement_rate", "ai_traffic_pct", "content_age_days",
                     "days_since_last_update", "word_count"]
categorical_features = ["content_type", "main_intent"]
print(f"{len(numeric_features)} numeric + {len(categorical_features)} categorical features, sklearn model-ready")

9 numeric + 2 categorical features, sklearn model-ready


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Split: grouped by `client_id`, not random and not time-based.** The data dictionary flags `client_id` explicitly for client-holdout splits — a random row split would let the model see some pages from every client during training and effectively memorize "how client X's pages behave," which is a form of the grouped-entity leakage the hunting-leakage skill warns about (rows from one group share hidden character). A time split isn't available here: this teaching slice is already a single 90-day-aggregated snapshot per page, with no per-row date to split on — that only becomes possible at the full-warehouse level (`fact_content_daily_performance`), which is out of scope for this lane's 30k-row file. So: `GroupShuffleSplit` on `client_id`, 75/25, seeded.

In [3]:
X = df[numeric_features + categorical_features]
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups))

print(f"Train: {len(train_idx):,} rows across {df.iloc[train_idx]['client_id'].nunique()} clients")
print(f"Test:  {len(test_idx):,} rows across {df.iloc[test_idx]['client_id'].nunique()} clients")
print(f"Train base rate: {y.iloc[train_idx].mean():.3f} | Test base rate: {y.iloc[test_idx].mean():.3f}")
print("No client appears in both sets:", set(df.iloc[train_idx]['client_id']).isdisjoint(set(df.iloc[test_idx]['client_id'])))

Train: 22,885 rows across 24 clients
Test:  7,115 rows across 8 clients
Train base rate: 0.550 | Test base rate: 0.517
No client appears in both sets: True


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Same test rows, three ways to score them:** the Week-4 baseline rule (refit its CTR benchmark on TRAIN clients only, to be fair), Logistic Regression, and Random Forest — all evaluated with the identical `precision_at_k` function on the identical held-out test rows.

In [4]:
# --- rebuild the Week-4 baseline, fitting its CTR benchmark on TRAIN clients only ---
train_df = df.iloc[train_idx]
bench_pool = train_df[(train_df["impressions_90d"] >= 100) & (train_df["avg_position"] > 0)]
bench = bench_pool.groupby("position_tier").apply(
    lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100
).to_dict()

def baseline_score(row):
    if row["impressions_90d"] < 100:
        return 0.0
    stale = min(row["days_since_last_update"], 365) / 365 * 40
    ctr_gap = 0.0
    if row["avg_position"] > 0:
        tb = bench.get(row["position_tier"])
        if tb:
            ctr_gap = min(max(0.0, tb - row["ctr"]) / tb, 1.0) * 60
    return stale + ctr_gap

df["baseline_score"] = df.apply(baseline_score, axis=1)

# --- train models on TRAIN only ---
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

lr_prep = ColumnTransformer([("num", StandardScaler(), numeric_features),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
lr = Pipeline([("prep", lr_prep), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_SEED))])
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]

rf_prep = ColumnTransformer([("num", "passthrough", numeric_features),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])
rf = Pipeline([("prep", rf_prep), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8,
                                                                    random_state=RANDOM_SEED, n_jobs=-1))])
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

test_df = df.iloc[test_idx]
test_base_rate = test_df["is_declining_label"].mean()

rows = []
for k in [20, 50, 100, 500]:
    rows.append({
        "k": k,
        "base_rate": round(test_base_rate, 3),
        "baseline_rule": round(precision_at_k(test_df["baseline_score"].values, test_df["is_declining_label"].values, k), 3),
        "logistic_regression": round(precision_at_k(lr_probs, y_test.values, k), 3),
        "random_forest": round(precision_at_k(rf_probs, y_test.values, k), 3),
    })
comparison_table = pd.DataFrame(rows)
print(comparison_table.to_string(index=False))

  k  base_rate  baseline_rule  logistic_regression  random_forest
 20      0.517          0.600                 0.80          0.550
 50      0.517          0.520                 0.66          0.600
100      0.517          0.550                 0.58          0.620
500      0.517          0.514                 0.53          0.626


**Reading the table:** Logistic Regression wins clearly at the top of the queue (precision@20 = 0.800 vs. the baseline's 0.600 and the 0.517 base rate) — its linear score separates the most obvious cases well. Random Forest starts weaker at k=20 (0.550) but pulls ahead as the queue gets longer (precision@500 = 0.626 vs. LR's 0.530 and the baseline's 0.514), suggesting it's picking up more nuanced, non-linear patterns further down the list. Neither model wins everywhere — I'm reporting both rather than picking one metric that flatters a single model, per the training-honest-models skill.

**Practical read:** if FlyRank's editors only ever look at the top ~20-50 flagged pages a week, Logistic Regression is the better pick. If the queue is meant to be worked further down (top ~500), Random Forest is worth the extra complexity.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:
# Feature importance sanity check -- is any single feature suspiciously dominant? (leakage smell)
importances = rf.named_steps["clf"].feature_importances_
ohe_cols = rf.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(categorical_features)
all_cols = numeric_features + list(ohe_cols)
imp = pd.Series(importances, index=all_cols).sort_values(ascending=False)
print("Random Forest feature importances (top 6):")
print(imp.head(6))

Random Forest feature importances (top 6):
impressions_90d           0.260336
content_age_days          0.211643
avg_position              0.200678
word_count                0.087074
days_since_last_update    0.049801
clicks_90d                0.049025
dtype: float64


**No leakage smell:** the top feature (`impressions_90d`) accounts for 26% of the Random Forest's importance, not the 90%+ 'towers over everything' pattern the hunting-leakage skill warns is a leakage confession. Importance is spread across `content_age_days` (21%), `avg_position` (20%), and `word_count` (9%) too — a believable mix of visibility, age, and ranking-quality signals, not one column secretly encoding the answer.

In [7]:
test_view = test_df.copy()
test_view["rf_prob"] = rf_probs

fp = test_view[test_view["is_declining_label"] == 0].sort_values("rf_prob", ascending=False).head(3)
fn = test_view[test_view["is_declining_label"] == 1].sort_values("rf_prob", ascending=True).head(3)

cols = ["content_id", "rf_prob", "impressions_90d", "avg_position", "ctr",
        "content_age_days", "days_since_last_update"]
print("Confident FALSE POSITIVES (model said high-risk, page was actually stable/growing):")
print(fp[cols].to_string(index=False))
print("\nConfident FALSE NEGATIVES (model said low-risk, page was actually declining):")
print(fn[cols].to_string(index=False))

Confident FALSE POSITIVES (model said high-risk, page was actually stable/growing):
          content_id  rf_prob  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update
content_3164f3076003 0.834834             2696          16.1 0.04               275                     104
content_4d9f36001f06 0.825554             3369          13.2 0.03               275                     104
content_884c401ce126 0.824525              101          23.1 0.00               174                      92

Confident FALSE NEGATIVES (model said low-risk, page was actually declining):
          content_id  rf_prob  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update
content_7bc32bc1df59 0.106227                1           0.0  0.0               238                      92
content_16f38acf0f26 0.163994                2          50.0  0.0               358                      20
content_df23b4bc766d 0.178736              665          49.6  0.0               5

**Where the model is wrong, and why that's plausible, not a bug:**

- **False positives** (rows 1-3) all sit at decent impressions (100-3,369), mid-table position (13-23), and moderate staleness (~92-104 days) — exactly the profile my Week-4 baseline and this model both treat as risky. But they didn't decline. My guess: these are pages sitting in a stable equilibrium at that position, where the model's learned pattern ("mid-position + a bit stale = risk") is a reasonable prior that just doesn't hold for every page — the model has no way to see *why* a specific page is stable there.
- **False negatives** (rows 1-3) all have very low `impressions_90d` (1, 2, and 665). This is the more interesting failure: `trend_direction` is a percentage-change label, and percentage change on a near-zero base is naturally noisy and volatile — a page going from 1 impression to 0 reads as a 100% decline even though nothing meaningful happened. The model has learned that low-traffic pages are usually safe (correctly, most of the time), which makes it miss the cases where the *label itself* is unstable at low volume. That's a limitation of the label definition, not something more training data alone would fix.

**What this model leans on:** visibility (`impressions_90d`), age, and position — sensible editorial signals, not something exotic or suspicious.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*